In [2]:
# Import shared loading/cleaning functions from utils.py
from utils import load_clean_data
from utils import get_cv_folds

import pandas as pd
import numpy as np

# category_encoders provides a scikit-learn-style transformer for frequency
# (count) encoding, so we no longer need to build the mapping by hand
import category_encoders as ce

# Models we will train and compare
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor

# For scaling numeric features (needed for Linear Regression and KNN)
from sklearn.preprocessing import StandardScaler

# For evaluation metrics
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# Note: Frequency Encoding is now done with category_encoders.CountEncoder
# instead of a manual pandas mapping. CountEncoder replaces each category with
# how many times it appears in the data it was fit on (i.e. the TRAINING data
# for each fold), which is exactly what the manual version did.

In [3]:
# Load the cleaned dataset
df = load_clean_data("data.csv")

# Generate the FIXED set of cross-validation folds — same folds as every
# other notebook, for valid paired comparison later
folds = get_cv_folds(df)

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Generated {len(folds)} folds")

Loaded 79159 rows, 17 columns
Generated 10 folds


In [4]:
# The target variable we're predicting
y = df["advertised_price"]

# Drop the target from the feature set
X = df.drop(columns=["advertised_price"])

# Categorical columns to encode
categorical_cols = ["make", "model", "generation", "fuel", "body",
                     "transmission", "trim", "plate_age_id"]

# Numeric columns — vehicle_age_days excluded (redundant with vehicle_age_years,
# confirmed via feature importance comparison in the One-Hot notebook)
numeric_cols = ["mileage", "list_price", "reg_year", "doors", "engine",
                "engine_missing", "vehicle_age_years"]

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")

Categorical columns: 8
Numeric columns: 7


In [5]:
# Reset results dictionary — storing RMSE, MAE, and R² for each model, per fold
results = {
    "Linear Regression": {"rmse": [], "mae": [], "r2": []},
    "KNN": {"rmse": [], "mae": [], "r2": []},
    "Decision Tree": {"rmse": [], "mae": [], "r2": []},
    "Random Forest": {"rmse": [], "mae": [], "r2": []},
    "Stacking": {"rmse": [], "mae": [], "r2": []}
}

In [6]:
# Loop through each of the 10 fixed folds
for fold_num, (train_idx, test_idx) in enumerate(folds, start=1):

    # ---- Split data into this fold's train/test portions ----
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # ---- Frequency Encode the categorical columns ----
    # CountEncoder is fit on TRAINING data only (this is the leakage-prevention
    # step — test data must never contribute to the counts), then used to
    # transform both train and test.
    # handle_unknown=0 means a category seen in test but not in training gets
    # a count of 0 — the same behaviour as the original .fillna(0) step.
    count_encoder = ce.CountEncoder(
        cols=categorical_cols,
        handle_unknown=0,
        handle_missing=0,
    )
    X_train_cat = count_encoder.fit_transform(X_train[categorical_cols])
    X_test_cat = count_encoder.transform(X_test[categorical_cols])

    # ---- Combine encoded categorical columns with numeric columns ----
    X_train_final = pd.concat([X_train[numeric_cols], X_train_cat], axis=1)
    X_test_final = pd.concat([X_test[numeric_cols], X_test_cat], axis=1)

    # ---- Scale features for models that need it (Linear Regression, KNN) ----
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final)
    X_test_scaled = scaler.transform(X_test_final)

    # ---- Define the 5 models ----
    linear_model = LinearRegression()
    knn_model = KNeighborsRegressor()
    tree_model = DecisionTreeRegressor(random_state=42)
    rf_model = RandomForestRegressor(random_state=42, n_estimators=50)

    stacking_model = StackingRegressor(
        estimators=[
            ("linear", LinearRegression()),
            ("knn", KNeighborsRegressor()),
            ("tree", DecisionTreeRegressor(random_state=42)),
            ("rf", RandomForestRegressor(random_state=42, n_estimators=50))
        ],
        final_estimator=LinearRegression(),
        cv=3,
    )

    # ---- Train and evaluate each model ----

    # Linear Regression (scaled features)
    linear_model.fit(X_train_scaled, y_train)
    linear_preds = linear_model.predict(X_test_scaled)
    results["Linear Regression"]["rmse"].append(root_mean_squared_error(y_test, linear_preds))
    results["Linear Regression"]["mae"].append(mean_absolute_error(y_test, linear_preds))
    results["Linear Regression"]["r2"].append(r2_score(y_test, linear_preds))

    # KNN (scaled features)
    knn_model.fit(X_train_scaled, y_train)
    knn_preds = knn_model.predict(X_test_scaled)
    results["KNN"]["rmse"].append(root_mean_squared_error(y_test, knn_preds))
    results["KNN"]["mae"].append(mean_absolute_error(y_test, knn_preds))
    results["KNN"]["r2"].append(r2_score(y_test, knn_preds))

    # Decision Tree (unscaled features)
    tree_model.fit(X_train_final, y_train)
    tree_preds = tree_model.predict(X_test_final)
    results["Decision Tree"]["rmse"].append(root_mean_squared_error(y_test, tree_preds))
    results["Decision Tree"]["mae"].append(mean_absolute_error(y_test, tree_preds))
    results["Decision Tree"]["r2"].append(r2_score(y_test, tree_preds))

    # Random Forest (unscaled features)
    rf_model.fit(X_train_final, y_train)
    rf_preds = rf_model.predict(X_test_final)
    results["Random Forest"]["rmse"].append(root_mean_squared_error(y_test, rf_preds))
    results["Random Forest"]["mae"].append(mean_absolute_error(y_test, rf_preds))
    results["Random Forest"]["r2"].append(r2_score(y_test, rf_preds))

    # Stacking (unscaled features)
    stacking_model.fit(X_train_final, y_train)
    stacking_preds = stacking_model.predict(X_test_final)
    results["Stacking"]["rmse"].append(root_mean_squared_error(y_test, stacking_preds))
    results["Stacking"]["mae"].append(mean_absolute_error(y_test, stacking_preds))
    results["Stacking"]["r2"].append(r2_score(y_test, stacking_preds))

    # Print progress so we can see the loop working fold by fold
    print(f"Fold {fold_num}/{len(folds)} complete")

print("\nAll folds complete.")

Fold 1/10 complete
Fold 2/10 complete
Fold 3/10 complete
Fold 4/10 complete
Fold 5/10 complete
Fold 6/10 complete
Fold 7/10 complete
Fold 8/10 complete
Fold 9/10 complete
Fold 10/10 complete

All folds complete.


In [7]:
# Build a summary table: mean and std of RMSE, MAE, and R² across the folds, per model
summary_rows = []
for model_name, metrics in results.items():
    summary_rows.append({
        "model": model_name,
        "mean_rmse": np.mean(metrics["rmse"]),
        "std_rmse": np.std(metrics["rmse"]),
        "mean_mae": np.mean(metrics["mae"]),
        "std_mae": np.std(metrics["mae"]),
        "mean_r2": np.mean(metrics["r2"]),
        "std_r2": np.std(metrics["r2"])
    })

summary = pd.DataFrame(summary_rows)
summary

,model,mean_rmse,std_rmse,mean_mae,std_mae,mean_r2,std_r2
0,Linear Regression,5086.867922,179.220047,3055.019090,40.277773,0.869054,0.006594
1,KNN,2611.641430,142.629814,1447.187909,25.950363,0.965458,0.003081
2,Decision Tree,2995.470355,195.412203,1608.408535,41.261844,0.954480,0.005497
3,Random Forest,2125.133644,143.695483,1161.871430,22.213491,0.977103,0.002679
4,Stacking,2134.181780,141.353597,1168.836183,21.187161,0.976910,0.002639


In [8]:
# Flatten results into a long-format DataFrame: one row per model per fold per metric
# Same structure as every other encoder notebook, so all files can be combined later
rows = []
for model_name, metrics in results.items():
    for fold_i in range(len(folds)):
        rows.append({
            "encoder": "frequency",
            "model": model_name,
            "fold": fold_i + 1,
            "rmse": metrics["rmse"][fold_i],
            "mae": metrics["mae"][fold_i],
            "r2": metrics["r2"][fold_i]
        })

results_df = pd.DataFrame(rows)
results_df.to_csv("frequency_scores.csv", index=False)

print("Saved results to frequency_scores.csv")

Saved results to frequency_scores.csv


In [9]:
# Capture feature importance from Random Forest, same as previous notebooks
feature_importance = pd.DataFrame({
    "feature": X_train_final.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance["encoder"] = "frequency"
feature_importance.to_csv("frequency_feature_importance.csv", index=False)

feature_importance.head(15)

,feature,importance,encoder
1,list_price,0.672685,frequency
6,vehicle_age_years,0.168737,frequency
0,mileage,0.092024,frequency
2,reg_year,0.012224,frequency
8,model,0.008190,frequency
13,trim,0.008175,frequency
14,plate_age_id,0.007869,frequency
7,make,0.007261,frequency
9,generation,0.006990,frequency
4,engine,0.006729,frequency
